# Интеллектуальный анализ данных – весна 2026
# Домашнее задание 6: классификация текстов

Правила:



*   Домашнее задание оценивается в 10 баллов.
*   Можно использовать без доказательства любые результаты, встречавшиеся на лекциях или семинарах по курсу, если получение этих результатов не является вопросом задания.
*  Можно использовать любые свободные источники с *обязательным* указанием ссылки на них.
*  Плагиат не допускается. При обнаружении случаев списывания, 0 за работу выставляется всем участникам нарушения, даже если можно установить, кто у кого списал.
*  Старайтесь сделать код как можно более оптимальным. В частности, будет штрафоваться использование циклов в тех случаях, когда операцию можно совершить при помощи инструментов библиотек, о которых рассказывалось в курсе.
* Если в задании есть вопрос на рассуждение, то за отсутствие ответа на него балл за задание будет снижен вполовину.

В этом домашнем задании вам предстоит построить классификатор текстов.

Будем предсказывать эмоциональную окраску твиттов о коронавирусе.



In [92]:
import numpy as np
import pandas as pd
from typing import  List
import matplotlib.pyplot as plt
import seaborn as sns
from string import punctuation

In [93]:
df = pd.read_csv('./content/tweets_coronavirus.csv', encoding='latin-1')
df.sample(4)

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
8124,13669,58621,"Long Beach, CA",20-03-2020,YÃÂall people need to stop panic buying! How...,Extremely Negative
18272,26031,70983,Somewhere Playing the Ponies,25-03-2020,"Gov Cuomo:""Trump not getting the much needed v...",Negative
1995,6228,51180,NaN,17-03-2020,@scottliving towels @amazon they are better th...,Extremely Positive
16983,24465,69417,World Wide,25-03-2020,Partners from AIMS International still connect...,Extremely Negative


Для каждого твитта указано:


*   UserName - имя пользователя, заменено на целое число для анонимности
*   ScreenName - отображающееся имя пользователя, заменено на целое число для анонимности
*   Location - местоположение
*   TweetAt - дата создания твитта
*   OriginalTweet - текст твитта
*   Sentiment - эмоциональная окраска твитта (целевая переменная)



## Задание 1 Подготовка (0.5 балла)

Целевая переменная находится в колонке `Sentiment`.  Преобразуйте ее таким образом, чтобы она стала бинарной: 1 - если у твитта положительная или очень положительная эмоциональная окраска и 0 - если отрицательная или очень отрицательная.

In [94]:
df.Sentiment.value_counts()

Sentiment
Positive              11422
Negative               9917
Extremely Positive     6624
Extremely Negative     5481
Name: count, dtype: int64

In [95]:
sentiment_map = {
    'Extremely Positive': 1,
    'Positive': 1,
    'Negative': 0,
    'Extremely Negative': 0
}

df['SentimentBin'] = df['Sentiment'].map(sentiment_map)

Сбалансированы ли классы?

In [96]:
df.SentimentBin.value_counts()

SentimentBin
1    18046
0    15398
Name: count, dtype: int64

**Ответ:** Да, я считаю что классы сбалансированы

Выведете на экран информацию о пропусках в данных. Если пропуски присутствуют заполните их строкой 'Unknown'.

In [97]:
df.isnull().sum()

UserName            0
ScreenName          0
Location         7049
TweetAt             0
OriginalTweet       0
Sentiment           0
SentimentBin        0
dtype: int64

In [98]:
df = df.fillna('Unknown')

In [99]:
df.isnull().sum()

UserName         0
ScreenName       0
Location         0
TweetAt          0
OriginalTweet    0
Sentiment        0
SentimentBin     0
dtype: int64

Разделите данные на обучающие и тестовые в соотношении 7 : 3 и укажите `random_state=0`

In [100]:
from sklearn.model_selection import train_test_split

X = df[[x for x in df.columns if x not in ['Sentiment', 'SentimentBin']]]
y = df['SentimentBin']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

In [101]:
X_train.shape

(23410, 5)

In [102]:
X_test.shape

(10034, 5)

## Задание 2 Токенизация (3 балла)

Постройте словарь на основе обучающей выборки и посчитайте количество встреч каждого токена с использованием самой простой токенизации - деления текстов по пробельным символам и приведения токенов в нижний регистр.

In [103]:
X_train.columns

Index(['UserName', 'ScreenName', 'Location', 'TweetAt', 'OriginalTweet'], dtype='str')

In [104]:
X_train.dtypes

UserName         int64
ScreenName       int64
Location           str
TweetAt            str
OriginalTweet      str
dtype: object

In [105]:
X_train.head()

,UserName,ScreenName,Location,TweetAt,OriginalTweet
25621,35178,80130,Unknown,06-04-2020,Why we still want to buy so much stuff during ...
30135,40819,85771,"Boston, MA HQ",10-04-2020,With driving even more usage a strong strategy...
28899,39249,84201,India,09-04-2020,@Canon_India I am very happy.. Great job by @C...
5989,11068,56020,"Mayfair, London, UK",19-03-2020,The U.S national debt will likely exceed $30 T...
4367,9109,54061,WNC,18-03-2020,"Finally got to the grocery store. Honestly, wh..."


In [106]:
from collections import defaultdict

In [107]:
def make_tokens(s: str):
    s = s.replace(
        ".", " "
    )
    s = s.replace(
        "!", " "
    )
    s = s.replace(
        ",", " "
    )
    s = s.replace(
        ";", " "
    )
    s = s.replace(
        "-", " "
    )
    s = s.replace(
        "'", " "
    )
    s = s.replace(
        "\"", " "
    )
    return s.lower().split()

all_tokens = X_train['Location'].apply(make_tokens).sum() + X_train['OriginalTweet'].apply(make_tokens).sum()
print(all_tokens[:10])

['unknown', 'boston', 'ma', 'hq', 'india', 'mayfair', 'london', 'uk', 'wnc', 'ellicott']


In [108]:
dictionary = defaultdict(int)
for el in all_tokens:
    dictionary[el] += 1

print(len(dictionary.keys()))

66598


Какой размер словаря получился?

In [109]:
print(len(dictionary.keys()))

66598


Выведите 10 самых популярных токенов с количеством встреч каждого из них. Объясните, почему именно эти токены в топе.

In [110]:
sorted_tokens = sorted(
    dictionary.items(), 
    key=lambda x: x[1], reverse=True
)

for i, (token, count) in enumerate(sorted_tokens[:10], 1):
    print(f"{token} : {count:>6}")

the :  27309
to :  23530
and :  14911
of :  13298
https://t :  12861
a :  11943
in :  11637
#coronavirus :   8661
for :   8614
is :   7478


**Ответ:** Это предлоги и языковые конструкции английского языка, они встечаются чаще всего, так как большинство грамматических конструкций их использует, вне зависимости от смысла

Удалите стоп-слова из словаря и выведите новый топ-10 токенов (и количество встреч) по популярности.  Что можно сказать  о нем?

In [111]:
%pip install nltk


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [112]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

filtered_dict = defaultdict(int)

for key, value in dictionary.items():
    if key not in stop_words:
        filtered_dict[key] = value

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/karlkorhonen/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [113]:
sorted_tokens = sorted(
    filtered_dict.items(), 
    key=lambda x: x[1], reverse=True
)

for i, (token, count) in enumerate(sorted_tokens[:10], 1):
    print(f"{token} : {count}")

https://t : 12861
#coronavirus : 8661
19 : 6582
covid : 6123
unknown : 4961
prices : 4521
food : 4314
store : 3826
supermarket : 3729
grocery : 3500


**Ответ:**  теперь видно, что самые популярные токены связаны с коронавирусом, за исключением ссылки, которая лишь косвенно связана, возможно там ссыль на какую-то новость

Также выведите 20 самых непопулярных слов (если самых непопулярных слов больше, выведите любые 20 из них) Почему эти токены непопулярны, требуется ли как-то дополнительно работать с ними?

In [114]:
sorted_tokens = sorted(
    filtered_dict.items(), 
    key=lambda x: x[1], reverse=False
)

for i, (token, count) in enumerate(sorted_tokens[:20], 1):
    print(f"{token} : {count}")

wnc : 1
ellicott : 1
il/halas : 1
nyc/washington : 1
downstage : 1
bonaventure : 1
beaconsfield : 1
momma : 1
63040 : 1
#torydystopia?#gtto#jc4ever? : 1
wiesbaden : 1
nevaland : 1
contiguous : 1
wilds : 1
geloãâs : 1
20850 : 1
stewing : 1
agonies : 1
gosport/portsmouth : 1
earth/matrix : 1


**Ответ:** слова бессмысленны, анализировать их не имеет смысла, так как они никак не указывают на эмоциональный окрас



Теперь воспользуемся токенайзером получше - TweetTokenizer из библиотеки nltk. Примените его и посмотрите на топ-10 популярных слов. Чем он отличается от топа, который получался раньше? Почему?

In [115]:
from nltk.tokenize import TweetTokenizer
from collections import defaultdict
import nltk

tweet_tokenizer = TweetTokenizer()
dictionary_tt = defaultdict(int)


for location in X_train['Location']:
    tokens = tweet_tokenizer.tokenize(str(location).lower())
    for token in tokens:
        dictionary_tt[token] += 1


for tweet in X_train['OriginalTweet']:
    tokens = tweet_tokenizer.tokenize(str(tweet).lower())
    for token in tokens:
        dictionary_tt[token] += 1

In [116]:
sorted_tokens = sorted(
    dictionary_tt.items(), 
    key=lambda x: x[1], reverse=True
)

for i, (token, count) in enumerate(sorted_tokens[:10], 1):
    print(f"{token} : {count}")

, : 28177
the : 27369
. : 24776
to : 23512
and : 14983
of : 13291
a : 11996
in : 11587
? : 11190
#coronavirus : 8808


**Ответ:** к стоп словам тут добавилась пунктуация (я ее ручками чистил в самодельном токенизаторе)

Удалите из словаря стоп-слова и пунктуацию, посмотрите на новый топ-10 слов с количеством встреч, есть ли теперь в нем что-то не похожее на слова?

In [117]:
from string import punctuation
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english')) # !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
dictionary_clean = defaultdict(int)

for token, count in dictionary_tt.items():
    if token not in punctuation and token not in stop_words and not all(c in punctuation for c in token):
        dictionary_clean[token] = count

In [118]:
sorted_tokens = sorted(
    dictionary_clean.items(), 
    key=lambda x: x[1], reverse=True
)

for i, (token, count) in enumerate(sorted_tokens[:10], 1):
    print(f"{token} : {count}")

#coronavirus : 8808
â : 7720
 : 7409
19 : 7170
covid : 6256
unknown : 4961
prices : 4601
 : 4398
food : 4372
store : 3879


**Ответ:** Уже вроде норм слова пошли, но все еще присутствует какой-то мусор в виде невидимых символов и странных буков

Скорее всего в некоторых топах были неотображаемые символы или отдельные буквы не латинского алфавита. Уберем их: удалите из словаря токены из одного символа, позиция которого в таблице Unicode 128 и более (`ord(x) >= 128`)

Выведите топ-10 самых популярных и топ-20 непопулярных слов. Чем полученные топы отличаются от итоговых топов, полученных при использовании токенизации по пробелам? Что теперь лучше, а что хуже?

In [119]:
from string import punctuation
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english')) # !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
dictionary_clean = defaultdict(int)

for token, count in dictionary_tt.items():
    if token not in punctuation and token not in stop_words and not all(c in punctuation for c in token) and all(ord(c) < 128 for c in token):
        dictionary_clean[token] = count

In [120]:
sorted_tokens = sorted(
    dictionary_clean.items(), 
    key=lambda x: x[1], reverse=True
)

for i, (token, count) in enumerate(sorted_tokens[:10], 1):
    print(f"{token} : {count}")

#coronavirus : 8808
19 : 7170
covid : 6256
unknown : 4961
prices : 4601
food : 4372
store : 3879
supermarket : 3805
grocery : 3523
people : 3464


In [127]:
sorted_tokens = sorted(
    dictionary_clean.items(), 
    key=lambda x: x[1], reverse=False
)

for i, (token, count) in enumerate(sorted_tokens[:20], 1):
    print(f"{token} : {count}")

wnc : 1
ellicott : 1
halas : 1
downstage : 1
bonaventure : 1
beaconsfield : 1
momma's : 1
63040 : 1
#torydystopia : 1
#gtto : 1
#jc4ever : 1
wiesbaden : 1
nevaland : 1
contiguous : 1
wilds : 1
20850 : 1
stewing : 1
agonies : 1
gosport : 1
washingron : 1


**Ответ:** # топы довольно похожи, но в первом топе (топ10 самых популярных) с первого места убралась ссылка. Топ непопулярных все еще очень похож на предыдущий (такой же мусор)

Выведите топ-10 популярных хештегов (токены, первые символы которых - #) с количеством встреч. Что можно сказать о них?

In [128]:
dictionary_clean_hashtag = {
    key : value for key, value in dictionary_clean.items() if key[0] == '#'
}

hashtag_tokens = sorted(
    dictionary_clean_hashtag.items(), 
    key=lambda x: x[1], reverse=True
)

for i, (token, count) in enumerate(hashtag_tokens[:20], 1):
    print(f"{token} : {count}")

#coronavirus : 8808
#covid19 : 2589
#covid_19 : 1734
#covid2019 : 946
#toiletpaper : 744
#covid : 641
#socialdistancing : 465
#coronacrisis : 448
#pandemic : 257
#coronaviruspandemic : 249
#stayhome : 235
#coronavirusoutbreak : 223
#covid-19 : 218
#corona : 209
#lockdown : 208
#supermarket : 206
#stayathome : 197
#panicbuying : 197
#stayhomesavelives : 194
#stophoarding : 190


**Ответ:** все хештеги так или иначе связаны с коронавирусом, имеют различные призывы в себе

То же самое проделайте для ссылок на сайт https://t.co Сравнима ли популярность ссылок с популярностью хештегов? Будет ли информация о ссылке на конкретную страницу полезна?

In [129]:
dictionary_clean_hashtag = {
    key : value for key, value in dictionary_clean.items() if key.startswith('https://t.co')
}

hashtag_tokens = sorted(
    dictionary_clean_hashtag.items(), 
    key=lambda x: x[1], reverse=True
)

for i, (token, count) in enumerate(hashtag_tokens[:20], 1):
    print(f"{token} : {count}")

https://t.co/oxa7swtond : 5
https://t.co/gp3eusapl8 : 4
https://t.co/wrlhyzizaa : 3
https://t.co/kuwipf1kqw : 3
https://t.co/zjnrx6dkkn : 3
https://t.co/3gbbdpdjat : 3
https://t.co/e2znxajpre : 3
https://t.co/catkegayoy : 3
https://t.co/g63rp042ho : 3
https://t.co/aziqcdgrnn : 3
https://t.co/bylqxrjmnt : 3
https://t.co/wuieefsnoj : 3
https://t.co/oi39zsanq8 : 3
https://t.co/rafj2l2ceq : 2
https://t.co/of8enb5kxp : 2
https://t.co/ozeniasfeo : 2
https://t.co/9gjx7znzf4 : 2
https://t.co/i7lk8xuvmr : 2
https://t.co/skwgk66i8z : 2
https://t.co/r7sagojsjg : 2


**Ответ:** мне кажется ссылки будут не оч полезны, ибо мы не можем просто так посмотреть через код что под ними находится, или это оч трудозатратно будет (слать запросы и т д)

Используем опыт предыдущих экспериментов и напишем собственный токенайзер, улучшив TweetTokenizer. Функция tokenize должна:



*   Привести текст в нижний регистр
*   Применить TweetTokenizer для  выделения токенов
*   Удалить стоп-слова, пунктуацию, токены из одного символа с позицией в таблице Unicode 128 и более,  ссылки на t.co



In [132]:
def custom_tokenizer(text):
  try:
    text = str(text).lower()
    tokens = tweet_tokenizer.tokenize(text)
    cleaned_tokens = []
    for token in tokens:
      if 't.co' in token or token.startswith('http') or token.startswith('www'):
        continue
      if token in stop_words or token in punctuation:
        continue
      if all(c in punctuation for c in token):
        continue
      if len(token) == 1:
        continue
      if any(ord(c) >= 128 for c in token):
        continue
      cleaned_tokens.append(token)
    return cleaned_tokens
  except TypeError as e:
    print(f'error while working with type {e}')
  except Exception as e:
    print(f'An exception was encountered {e}')


In [134]:
custom_tokenizer('This is sample text!!!! @Sample_text I, \x92\x92 https://t.co/sample  #sampletext')

['sample', 'text', '@sample_text', '#sampletext']

## Задание 3 Векторизация текстов (2 балла)

Обучите CountVectorizer с использованием custom_tokenizer в качестве токенайзера. Как размер полученного словаря соотносится с размером изначального словаря из начала задания 2?

In [126]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer # -- YOUR CODE HERE --

print(len(cv.vocabulary_))

AttributeError: type object 'CountVectorizer' has no attribute 'vocabulary_'

**Ответ:** # -- YOUR ANSWER HERE --

Посмотрим на какой-нибудь конкретный твитт:

In [ ]:
ind = 9023
train.iloc[ind]['OriginalTweet'], train.iloc[ind]['Sentiment']

Автор твитта не доволен ситуацией с едой во Франции и текст имеет резко негативную окраску.

Примените обученный CountVectorizer для векторизации данного текста, и попытайтесь определить самый важный токен и самый неважный токен (токен, компонента которого в векторе максимальна/минимальна, без учета 0). Хорошо ли они определились, почему?

In [ ]:
# -- YOUR CODE HERE --

**Ответ:** # -- YOUR ANSWER HERE --

Теперь примените TfidfVectorizer и  определите самый важный/неважный токены. Хорошо ли определились, почему?

In [ ]:
# -- YOUR CODE HERE --

**Ответ:** # -- YOUR ANSWER HERE --

Найдите какой-нибудь положительно окрашенный твитт, где TfidfVectorizer хорошо (полезно для определения окраски) выделяет важный токен, поясните пример.

*Подсказка:* явно положительные твитты можно искать при помощи положительных слов (good, great, amazing и т. д.)

In [ ]:
train[train['OriginalTweet'].apply(lambda x: 'your_good_word_here' in x) & (train['Sentiment'] == 1)]

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment


In [ ]:
# -- YOUR CODE HERE --

**Ответ:** # -- YOUR ANSWER HERE --

## Задание 4 Обучение первых моделей (1 балл)

Примените оба векторайзера для получения матриц с признаками текстов.  Выделите целевую переменную.

In [ ]:
# -- YOUR CODE HERE --

Обучите логистическую регрессию на векторах из обоих векторайзеров. Посчитайте долю правильных ответов на обучающих и тестовых данных. Какой векторайзер показал лучший результат? Что можно сказать о моделях?

Используйте `sparse` матрицы (после векторизации), не превращайте их в `numpy.ndarray` или `pd.DataFrame` - может не хватить памяти.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# -- YOUR CODE HERE --

**Ответ:** # -- YOUR ANSWER HERE --

## Задание 5 Стемминг (0.5 балла)

Для уменьшения словаря можно использовать стемминг.

Модифицируйте написанный токенайзер, добавив в него стемминг с использованием SnowballStemmer. Обучите Count- и Tfidf- векторайзеры. Как изменился размер словаря?

In [ ]:
def custom_stem_tokenizer(text):
  # -- YOUR CODE HERE --
  return tokens

In [ ]:
custom_stem_tokenizer('This is sample text!!!! @Sample_text I, \x92\x92 https://t.co/sample  #sampletext adding more words to check stemming')

['sampl', 'text', '@sample_text', '#sampletext', 'ad', 'word', 'check', 'stem']

In [ ]:
cv = CountVectorizer # -- YOUR CODE HERE --

print(len(cv.vocabulary_))

36652


**Ответ** # -- YOUR ANSWER HERE --

Обучите логистическую регрессию с использованием обоих векторайзеров. Изменилось ли качество? Есть ли смысл применять стемминг?

In [ ]:
# -- YOUR CODE HERE --

**Ответ:** # -- YOUR ANSWER HERE --

## Задание  6 Работа с частотами (1.5 балла)

Еще один способ уменьшить количество признаков - это использовать параметры min_df и max_df при построении векторайзера  эти параметры помогают ограничить требуемую частоту встречаемости токена в документах.

По умолчанию берутся все токены, которые встретились хотя бы один раз.



Подберите max_df такой, что размер словаря будет 36651 (на 1 меньше, чем было). Почему параметр получился такой большой/маленький?

In [ ]:
cv_df = CountVectorizer(tokenizer=custom_stem_tokenizer,
                        max_df=# -- YOUR CODE HERE --
                        ).fit(
                            # -- YOUR CODE HERE --
                            )
print(len(cv_df.vocabulary_))

36651


In [ ]:
# -- YOUR CODE HERE --

**Ответ:** # -- YOUR ANSWER HERE --

Подберите min_df (используйте дефолтное значение max_df) в CountVectorizer таким образом, чтобы размер словаря был 3700 токенов (при использовании токенайзера со стеммингом), а качество осталось таким же, как и было. Что можно сказать о результатах?

In [ ]:
cv_df = CountVectorizer(tokenizer=custom_stem_tokenizer,
                        min_df=# -- YOUR CODE HERE --
                        ).fit(
                            # -- YOUR CODE HERE --
                            )
print(len(cv_df.vocabulary_))

3700


In [ ]:
# -- YOUR CODE HERE --

**Ответ:** # -- YOUR ANSWER HERE --

В предыдущих заданиях признаки не скалировались. Отскалируйте данные (при словаре размера 3.7 тысяч, векторизованные CountVectorizer), обучите логистическую регрессию, посмотрите качество и выведите `barplot`, содержащий по 10 токенов, с наибольшим по модулю положительными/отрицательными весами. Что можно сказать об этих токенах?

In [ ]:
from sklearn.preprocessing import StandardScaler
# -- YOUR CODE HERE --

**Ответ:** # -- YOUR ANSWER HERE --

## Задание 7 Другие признаки (1.5 балла)

Мы были сконцентрированы на работе с текстами твиттов и не использовали другие признаки - имена пользователя, дату и местоположение

Изучите признаки UserName и ScreenName. полезны ли они? Если полезны, то закодируйте их, добавьте к матрице с отскалированными признаками, обучите логистическую регрессию, замерьте качество.

In [ ]:
# -- YOUR CODE HERE --

**Ответ:** # -- YOUR ANSWER HERE --

Изучите признак TweetAt в обучающей выборке: преобразуйте его к типу datetime и нарисуйте его гистограмму с разделением по цвету на основе целевой переменной. Полезен ли он? Если полезен, то закодируйте его, добавьте к матрице с отскалированными признаками, обучите логистическую регрессию, замерьте качество.

In [ ]:
# -- YOUR CODE HERE --

**Ответ:** # -- YOUR ANSWER HERE --



Поработайте с признаком Location в обучающей выборке. Сколько уникальных значений?

In [ ]:
# -- YOUR CODE HERE --

Постройте гистограмму топ-10 по популярности местоположений (исключая Unknown)

In [ ]:
# -- YOUR CODE HERE --

Видно, что многие местоположения включают в себя более точное название места, чем другие (Например, у некоторых стоит London, UK; а у некоторых просто UK или United Kingdom).

Создайте новый признак WiderLocation, который содержит самое широкое местоположение (например, из London, UK должно получиться UK). Сколько уникальных категорий теперь? Постройте аналогичную гистограмму.

In [ ]:
# -- YOUR CODE HERE --

Закодируйте признак WiderLocation с помощью OHE таким образом, чтобы создались только столбцы для местоположений, которые встречаются более одного раза. Сколько таких значений?


In [ ]:
# -- YOUR CODE HERE --

Добавьте этот признак к матрице отскалированных текстовых признаков, обучите логистическую регрессию, замерьте качество. Как оно изменилось? Оказался ли признак полезным?


*Подсказка:* используйте параметр `categories` в энкодере.

In [ ]:
# -- YOUR CODE HERE --

**Ответ:** # -- YOUR ANSWER HERE --